# 00 · Variable cible et table fusionnée

**Responsables :** _à compléter_

Ce notebook produit la table commune à tous les axes : `data/processed/morceaux.parquet`, une ligne par morceau, avec la cible `succes` et la clé `join_key`.

**À trancher en groupe avant de coder :**
- Fenêtre d'années commune (proposition : 2000–2023).
- Définition du succès : binaire « classé au Hot 100 / jamais classé », ou « top 10 parmi les classés ».
- Les **négatifs** : titres des datasets Spotify/Genius sortis sur la même période et jamais classés. Sans eux, biais du survivant.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from hit_factors import DATA_RAW, DATA_INTERIM, DATA_PROCESSED

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
from hit_factors import join_key
from hit_factors.data import load_billboard

## 1. Billboard Hot 100

In [ ]:
billboard = load_billboard()
billboard.head()

In [ ]:
# Une ligne par morceau : meilleur rang, nb de semaines, première semaine classée
hits = (
    billboard.groupby(["title", "performer"], as_index=False)
    .agg(peak=("peak_pos", "min"), semaines=("wks_on_chart", "max"), premiere_semaine=("chart_week", "min"))
)
hits["join_key"] = [join_key(t, a) for t, a in zip(hits["title"], hits["performer"])]
hits.describe(include="all")

## 2. Datasets Spotify (positifs + négatifs)

Téléchargement : voir `data/README.md`.

In [ ]:
# spotify = pd.read_csv(DATA_RAW / "spotify_tracks.csv")
# spotify["join_key"] = [join_key(t, a) for t, a in zip(spotify["track_name"], spotify["artists"])]

## 3. Jointure et cible

Jointure exacte sur `join_key`, puis fuzzy matching (`rapidfuzz`) pour les restants. **Mesurer et noter le taux de correspondance** : c'est une limite à présenter en soutenance.

In [ ]:
# morceaux = spotify.merge(hits, on="join_key", how="left")
# morceaux["succes"] = morceaux["peak"].notna()
# morceaux["succes"].value_counts(normalize=True)

In [ ]:
# morceaux.to_parquet(DATA_PROCESSED / "morceaux.parquet", index=False)